## Expansion of captions to prompts
#### Expand the generated captions to n number of prompts

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device) 

def parse_prompts(text, fallback_caption, n_expected):
    lines = [line.strip("-• \t") for line in text.splitlines()]
    prompts = [line for line in lines if len(line) > 3]

    if not prompts:
        prompts = re.findall(r'"([^"]+)"', text)

    prompts = [p for p in prompts if p]
    if not prompts:
        prompts = [fallback_caption]

    if len(prompts) < n_expected:
        prompts = (prompts * n_expected)[:n_expected]
    return prompts[:n_expected]

def refine_captions_local(captions, n_per_caption=3, max_new_tokens=120, temperature=0.9):
    print("Loaded refinement model:", MODEL, "on device:", device)
    prompts = []
    for c in captions:
        instruction = (
            f"You are an expert prompt engineer for image-generation pipelines (DreamShaper/LCM)."
            f"Refine this image caption into {n_per_caption} concise, high-quality image-generation prompts.\n"
            f"Caption: {c}\n\n"
            "Return one prompt per line. Do not use JSON."
        )
        inputs = tokenizer(instruction, return_tensors="pt", truncation=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            max_new_tokens=max_new_tokens,
            num_return_sequences=1,
        )
        text = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        prompts.extend(parse_prompts(text, c, n_per_caption))
    return prompts

In [ ]:
print("Generated captions for target images:")
for target_path in target_images:
    captions = generate_captions_for_image(target_path, num_captions=3)
    print(f"{Path(target_path).name} original captions:")
    for caption in captions:
        print("  -", caption)
    refined_prompts = refine_captions_local(captions, n_per_caption=3)
    print(f"{Path(target_path).name} refined prompts:")
    for prompt in refined_prompts:
        print("  -", prompt)